# RNN/LSTM/GRU (Recurrent Neural Networks)

이 노트북에서는 순환 신경망의 원리를 이해하고 시퀀스 데이터를 처리하는 모델을 구현합니다.

## 학습 목표
1. RNN의 원리 이해
2. LSTM과 GRU 아키텍처
3. 텍스트 생성 모델 구현
4. 시퀀스 분류

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. RNN 기본 개념

RNN은 시퀀스 데이터를 처리하기 위한 신경망입니다. 이전 시점의 정보를 현재 시점으로 전달합니다.

In [ ]:
# RNN 셀 동작 이해
input_size = 10
hidden_size = 20
seq_length = 5
batch_size = 3

# RNN 레이어
rnn = nn.RNN(input_size, hidden_size, batch_first=True)

# 입력 시퀀스
x = torch.randn(batch_size, seq_length, input_size)

# 순전파
output, hidden = rnn(x)

print(f"입력 shape: {x.shape}")
print(f"출력 shape: {output.shape}")
print(f"은닉 상태 shape: {hidden.shape}")
print(f"\n출력의 마지막 시점 = 최종 은닉 상태: {torch.allclose(output[:, -1, :], hidden.squeeze())}")

In [ ]:
# RNN vs LSTM vs GRU 비교
print("=" * 50)
print("RNN, LSTM, GRU 비교")
print("=" * 50)

rnn = nn.RNN(input_size, hidden_size, batch_first=True)
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
gru = nn.GRU(input_size, hidden_size, batch_first=True)

print(f"\nRNN 파라미터: {sum(p.numel() for p in rnn.parameters()):,}")
print(f"LSTM 파라미터: {sum(p.numel() for p in lstm.parameters()):,}")
print(f"GRU 파라미터: {sum(p.numel() for p in gru.parameters()):,}")

# LSTM은 hidden과 cell state를 반환
x = torch.randn(batch_size, seq_length, input_size)
lstm_out, (lstm_hidden, lstm_cell) = lstm(x)
print(f"\nLSTM hidden shape: {lstm_hidden.shape}")
print(f"LSTM cell shape: {lstm_cell.shape}")

## 2. 텍스트 생성 데이터 준비

In [ ]:
# 샘플 텍스트
text = """To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep, perchance to dream—ay, there's the rub."""

# 문자 집합 생성
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

vocab_size = len(chars)
print(f"텍스트 길이: {len(text)}")
print(f"어휘 크기: {vocab_size}")
print(f"문자 집합: {''.join(chars)}")

In [ ]:
class CharDataset(Dataset):
    def __init__(self, text, seq_length):
        self.text = text
        self.seq_length = seq_length
        self.data = [char_to_idx[ch] for ch in text]
    
    def __len__(self):
        return len(self.data) - self.seq_length
    
    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx+self.seq_length])
        y = torch.tensor(self.data[idx+1:idx+self.seq_length+1])
        return x, y

seq_length = 50
dataset = CharDataset(text, seq_length)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

x, y = dataset[0]
print(f"입력 shape: {x.shape}")
print(f"타겟 shape: {y.shape}")
print(f"\n입력 텍스트: {''.join([idx_to_char[i.item()] for i in x])}")
print(f"타겟 텍스트: {''.join([idx_to_char[i.item()] for i in y])}")

## 3. LSTM 텍스트 생성 모델

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.5):
        super(CharLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden=None):
        embed = self.embedding(x)
        output, hidden = self.lstm(embed, hidden)
        output = self.dropout(output)
        output = self.fc(output)
        return output, hidden
    
    def init_hidden(self, batch_size):
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        return (h0, c0)

model = CharLSTM(vocab_size, embed_size=64, hidden_size=128, num_layers=2).to(device)
print(model)
print(f"\n파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 모델 학습
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

losses = []
epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        output, _ = model(x)
        loss = criterion(output.view(-1, vocab_size), y.view(-1))
        loss.backward()
        
        # Gradient clipping
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

In [ ]:
# 학습 곡선
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

## 4. 텍스트 생성

In [ ]:
def generate_text(model, seed_text, length=200, temperature=1.0):
    model.eval()
    
    # 시드 텍스트를 인덱스로 변환
    chars = [char_to_idx.get(ch, 0) for ch in seed_text]
    input_seq = torch.tensor([chars]).to(device)
    
    generated = seed_text
    hidden = None
    
    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_seq, hidden)
            
            # Temperature scaling
            output = output[:, -1, :] / temperature
            probs = torch.softmax(output, dim=-1)
            
            # 샘플링
            next_char_idx = torch.multinomial(probs, 1).item()
            next_char = idx_to_char[next_char_idx]
            
            generated += next_char
            input_seq = torch.tensor([[next_char_idx]]).to(device)
    
    return generated

# 다양한 temperature로 텍스트 생성
print("=" * 60)
print("Temperature = 0.5 (더 결정적)")
print("=" * 60)
print(generate_text(model, "To be", length=150, temperature=0.5))

print("\n" + "=" * 60)
print("Temperature = 1.0 (균형)")
print("=" * 60)
print(generate_text(model, "To be", length=150, temperature=1.0))

print("\n" + "=" * 60)
print("Temperature = 1.5 (더 창의적)")
print("=" * 60)
print(generate_text(model, "To be", length=150, temperature=1.5))

## 5. 양방향 LSTM (Bidirectional LSTM)

In [ ]:
class BiLSTMClassifier(nn.Module):
    """양방향 LSTM 분류기"""
    def __init__(self, vocab_size, embed_size, hidden_size, num_classes, num_layers=2):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers,
                           batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        embed = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embed)
        
        # 양방향 hidden state 연결
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        
        out = self.dropout(hidden)
        out = self.fc(out)
        return out

bilstm = BiLSTMClassifier(vocab_size=1000, embed_size=128, hidden_size=256, num_classes=2)
print(bilstm)
print(f"\n파라미터 수: {sum(p.numel() for p in bilstm.parameters()):,}")

## 연습 문제

1. **GRU 모델**: LSTM 대신 GRU를 사용하는 모델을 구현하고 비교해보세요.
2. **다른 텍스트**: 다른 텍스트 데이터 (예: 한국어 소설)로 학습해보세요.
3. **시퀀스 길이**: 시퀀스 길이를 변경하고 결과를 비교해보세요.
4. **Attention**: 간단한 Self-Attention을 추가해보세요.

## 다음 단계

- [04_transformer.ipynb](./04_transformer.ipynb): Transformer 아키텍처 구현